# Phase 7A — Final Model Training (Google Colab)

Uses the **merged `main` implementation** to create the deployable balanced-XGBoost + sigmoid-calibration artifact.

**Holdout restriction:** the original 1,409-row historical holdout is permanently excluded from fitting, calibration, threshold selection, evaluation, and model selection. The repository's `prepare_training_partition()` reconstructs the original seed-42 stratified split and discards the non-training rows immediately. No holdout metrics are produced.

Outputs: `customer_churn_model.joblib` and `training_metadata.json`.


## 1. Runtime and exact repository source

Use a Colab Python 3.11 or 3.12 runtime, matching the repository's declared support. The notebook clones the latest `main` branch and records its commit SHA. It does not change or push repository files.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
REPO_DIR = Path("/content/customer-churn-prediction")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", PROJECT_COMMIT)


## 2. Install dependencies and provide the original CSV

Upload the original IBM Telco file only if it is not already present in this Colab runtime. It must retain the exact filename shown below.


In [ ]:
from google.colab import files
DATA_PATH = REPO_DIR / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    uploaded = files.upload()
    if DATA_PATH.name not in uploaded:
        raise FileNotFoundError(f"Upload the original file named {DATA_PATH.name}")
    DATA_PATH.write_bytes(uploaded[DATA_PATH.name])
print("Dataset:", DATA_PATH, "| bytes:", DATA_PATH.stat().st_size)


## 3. Structural preflight and import merged Phase 7A code

This preflight checks only file structure. It does not create a holdout split or calculate metrics. Split isolation, calibration, threshold selection, and fitting are handled by the committed source.


In [ ]:
import pandas as pd
raw_check = pd.read_csv(DATA_PATH)
expected = {
    "customerID","gender","SeniorCitizen","Partner","Dependents","tenure","PhoneService",
    "MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies","Contract","PaperlessBilling",
    "PaymentMethod","MonthlyCharges","TotalCharges","Churn"
}
if raw_check.shape != (7043, 21) or set(raw_check.columns) != expected:
    raise ValueError(f"Unexpected original dataset shape/schema: {raw_check.shape}")
if raw_check["customerID"].duplicated().any() or raw_check["Churn"].isna().any():
    raise ValueError("Duplicate IDs or missing target labels; refusing to train.")
if set(raw_check["Churn"].unique()) != {"Yes", "No"}:
    raise ValueError("Unexpected target labels.")
print("Original dataset structure verified:", raw_check.shape)
del raw_check

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.final_model import (
    CALIBRATION_METHOD, EXPECTED_DATASET_ROWS, EXPECTED_TRAINING_ROWS,
    METADATA_FILENAME, MODEL_FILENAME, train_final_model
)
from src.inference import load_model_artifact, predict_customer
from src.preprocess import FEATURE_COLUMNS
assert EXPECTED_DATASET_ROWS == 7043 and EXPECTED_TRAINING_ROWS == 5634
assert CALIBRATION_METHOD == "sigmoid" and len(FEATURE_COLUMNS) == 19
print("Merged Phase 7A source imported. Training rows:", EXPECTED_TRAINING_ROWS)


## 4. Train and export the final model

The merged `train_final_model()` performs the approved raw stratified split (seed 42), discards the historical holdout before cleaning, performs five-fold training-only OOF sigmoid calibration, selects a **fresh** F1 threshold, refits the base pipeline on all 5,634 training rows, and saves the bundle plus metadata. The old Phase 4 threshold is not reused.


In [ ]:
MODELS_DIR = REPO_DIR / "models"
ARTIFACT_PATH = MODELS_DIR / MODEL_FILENAME
METADATA_PATH = MODELS_DIR / METADATA_FILENAME
training_metadata = train_final_model(
    data_path=DATA_PATH,
    artifact_path=ARTIFACT_PATH,
    metadata_path=METADATA_PATH,
)
print("Training completed.")
print("Fresh training-OOF threshold:", training_metadata["threshold_selection"]["threshold"])
threshold_metadata = training_metadata.get("threshold_selection")
if not isinstance(threshold_metadata, dict):
    raise RuntimeError("Training completed, but threshold_selection metadata is missing or invalid.")

required_threshold_fields = {"threshold", "training_oof_f1_at_selected_threshold"}
missing_threshold_fields = required_threshold_fields - set(threshold_metadata)
if missing_threshold_fields:
    raise RuntimeError(f"Training metadata is missing required fields: {sorted(missing_threshold_fields)}")

selected_oof_f1 = float(threshold_metadata["training_oof_f1_at_selected_threshold"])
if not 0.0 <= selected_oof_f1 <= 1.0:
    raise RuntimeError(f"Unexpected training OOF F1 value: {selected_oof_f1}")
print("Training-OOF F1 used for threshold selection:", selected_oof_f1)
print("Artifact bytes:", training_metadata["artifact_size_bytes"])


## 5. Verify metadata and artifact provenance

These are metadata assertions only; they do not evaluate or score the historical holdout.


In [ ]:
import json
with METADATA_PATH.open(encoding="utf-8") as f:
    saved_metadata = json.load(f)
assert saved_metadata["artifact_version"] == 1
assert saved_metadata["dataset"]["rows"] == 7043 and saved_metadata["dataset"]["feature_count"] == 19
assert saved_metadata["split"]["training_rows"] == 5634
assert saved_metadata["split"]["historical_holdout_rows"] == 1409
assert saved_metadata["split"]["holdout_labels_used_only_for_stratification"] is True
assert saved_metadata["split"]["holdout_features_transformed"] is False
assert saved_metadata["split"]["holdout_features_used_for_fitting"] is False
assert saved_metadata["split"]["holdout_scored"] is False
assert saved_metadata["split"]["holdout_used_for_model_selection"] is False
assert saved_metadata["provenance"]["training_rows_only"] is True
assert saved_metadata["provenance"]["historical_holdout_scored"] is False
assert saved_metadata["provenance"]["historical_holdout_used_for_fitting"] is False
assert saved_metadata["calibration"]["method"] == "sigmoid"
assert saved_metadata["calibration"]["cv_splits"] == 5
assert saved_metadata["model"]["candidate"] == "Phase 4 XGBoost (balanced)"
assert 0 <= saved_metadata["threshold_selection"]["threshold"] <= 1
assert "training_oof_f1_at_selected_threshold" in saved_metadata["threshold_selection"]
assert 0 <= saved_metadata["threshold_selection"]["training_oof_f1_at_selected_threshold"] <= 1
assert saved_metadata["threshold_selection"]["training_oof_f1_at_selected_threshold"] == training_metadata["threshold_selection"]["training_oof_f1_at_selected_threshold"]
assert ARTIFACT_PATH.stat().st_size == saved_metadata["artifact_size_bytes"]
print("Metadata checks passed.")
print("Threshold:", saved_metadata["threshold_selection"]["threshold"])
print("Calibration:", saved_metadata["calibration"])
print("Artifact bytes:", saved_metadata["artifact_size_bytes"])


## 6. Reload and smoke-test inference

The following tests use synthetic customer rows only. They verify serialization and the single/batch inference interface, not predictive performance.


In [ ]:
loaded = load_model_artifact(ARTIFACT_PATH)
assert loaded["artifact_version"] == 1
assert loaded["calibration_method"] == "sigmoid"
assert loaded["threshold"] == saved_metadata["threshold_selection"]["threshold"]

synthetic = {
    "tenure":12, "MonthlyCharges":70.0, "TotalCharges":840.0,
    "gender":"Female", "SeniorCitizen":"No", "Partner":"Yes", "Dependents":"No",
    "PhoneService":"Yes", "MultipleLines":"No", "InternetService":"DSL",
    "OnlineSecurity":"No", "OnlineBackup":"Yes", "DeviceProtection":"No",
    "TechSupport":"No", "StreamingTV":"No", "StreamingMovies":"No",
    "Contract":"Month-to-month", "PaperlessBilling":"Yes",
    "PaymentMethod":"Electronic check"
}
single = predict_customer(synthetic, loaded)
assert set(single) == {"churn_probability","predicted_churn","decision_threshold"}
assert 0 <= single["churn_probability"] <= 1
batch = pd.DataFrame([synthetic, {**synthetic, "tenure":48, "MonthlyCharges":95.0,
    "TotalCharges":4560.0, "Contract":"Two year",
    "PaymentMethod":"Credit card (automatic)"}])
batch_result = predict_customer(batch, loaded)
assert len(batch_result) == 2 and batch_result["churn_probability"].between(0,1).all()
print("Serialization, single-row, and batch inference smoke tests passed.")
display(batch_result)


## 7. Download the two output files

Upload both generated files back to the project chat for review. Do not upload the raw CSV or commit the model artifact to Git.


In [ ]:
assert ARTIFACT_PATH.is_file() and METADATA_PATH.is_file()
print("Downloading:", ARTIFACT_PATH.name, "and", METADATA_PATH.name)
files.download(str(ARTIFACT_PATH))
files.download(str(METADATA_PATH))


## Completion checklist

- [ ] Run every cell from top to bottom.
- [ ] Confirm all metadata and inference assertions pass.
- [ ] Download `customer_churn_model.joblib`.
- [ ] Download `training_metadata.json`.
- [ ] Upload both files here for review.

**Interpretation:** the training OOF F1 is used for threshold selection, not an independent generalization estimate. The historical holdout is never evaluated.
